In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Composing the senses: dispatch, mimic, and the live UI

Everything is in place: eyes on the NPU, fingers on the CPU, a brain on the
iGPU, and topics carrying frames, joints, and intents. Component 3 is the
part that makes it a *robot* instead of a parts bin: composition.

![Alt Text](../../animations/modular_vla_pipeline.gif)

**Matching TODOs:** `3.1`-`3.4` in `project/integration/`. The harness for
this component is not a test script. It's the live UI (`app.py`).

In [ ]:
# workshop bootstrap: run against the finished reference code in solution/
# (no TODO stubs), so every cell works out of the box. To exercise YOUR
# implementations instead, change "solution" to "project" and restart.
import sys, pathlib
import time

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
assert (
    PROJECT / "config" / "workshop.yaml"
).exists(), f"solution not found at {PROJECT}"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from common.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from common.config import load_config, resolve, PROJECT_ROOT
import cv2, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
from common.motion import DryRunArm, clamp_joints, make_arm_backend, mapper_from_config
from common.fixtures import ScriptedPose, ScriptedHands, SyntheticCamera
from common.imaging import letterbox, decode_pose
from common.ros_helpers import ros2_available
from integration.dispatch import WorkshopContext
from common.quiet import QuietHands, quiet_stderr  # shared hush plumbing
from common.imaging import pick_arm
from common.imaging import draw_skeleton, draw_arm_panel, annotate_lines
from common.intents import CommandRegistry, Intent, ParsedCommand, is_stop, is_noise
from common.fixtures import offline_parse_text
from common.motion import DANCE_MOVES

cfg = load_config()
print("running against:", PROJECT_ROOT)

In [ ]:
# Inline image display helper (BGR -> notebook).


def show(img_bgr, title="", figsize=(9, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

## 1. Build a context: the bag every behavior receives

Handlers all have one signature: `handler(ctx, cmd)`. The `WorkshopContext`
carries the arm, the models, the camera, the mapper, shared state, and the
stop event. The next cell wires it the way `app.py` does: **try the real
thing, fall back a step at a time, and print which you got**. The models
come straight from the `models/` package (the bootstrap points at
`solution/`, so it is complete), with the notebook's own inline session
and an injected MediaPipe tracker as belt-and-braces fallbacks. Only with
no camera or model files at all does a slot degrade to a scripted
stand-in.

One taming step on top: the mapper is wrapped so `shoulder_lift` and
`elbow_flex` move as one **locked, opposed pair at 50% authority** - the
same link the 02_robot sliders default to - so the mimic can't throw the
forearm around at full reach.


In [ ]:
class DirectCamera:
    """Minimal camera client over cv2 - same .read()/.close() API as the
    stand-ins and the ROS client, so behaviors can't tell the difference."""

    def __init__(self, device, width=640, height=480, fps=30):
        self.cap = (
            cv2.VideoCapture(device, cv2.CAP_V4L2)  # V4L2 direct: no
            if isinstance(device, str)  # CV_IMAGES probing
            else cv2.VideoCapture(device)
        )
        if not self.cap.isOpened():
            self.cap.release()
            raise RuntimeError(f"cannot open camera {device}")
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, int(width))
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, int(height))
        self.cap.set(cv2.CAP_PROP_FPS, int(fps))

    def read(self):
        ok, frame = self.cap.read()
        return (ok, frame if ok else None)

    def close(self):
        self.cap.release()


class InlinePose:
    """YOLOv26s-pose on an inline ONNX session - the exact session-factory
    pattern from notebooks/01_ai_models/01_npu_yolo.ipynb, so the real model
    runs here even before your models/npu.py TODOs are done."""

    def __init__(self, cfg):
        import onnxruntime as ort

        y = cfg["yolo_pose"]
        onnx_path = resolve(y["onnx"])
        if not onnx_path.exists():
            raise FileNotFoundError(f"{onnx_path} not found")
        self.imgsz = int(y["imgsz"])
        self.conf_threshold = float(y["conf_threshold"])
        self.kpt_threshold = float(y["kpt_threshold"])
        device = y["device"]
        if (
            device == "npu"
            and "VitisAIExecutionProvider" not in ort.get_available_providers()
        ):
            device = "cpu"
        if device == "cpu":
            self.session = ort.InferenceSession(
                str(onnx_path), providers=["CPUExecutionProvider"]
            )
        else:
            cache = resolve(cfg["system"]["cache_dir"])
            cache.mkdir(parents=True, exist_ok=True)
            self.session = ort.InferenceSession(
                str(onnx_path),
                providers=["VitisAIExecutionProvider"],
                provider_options=[
                    {
                        "config_file": str(resolve(y["vitisai_config"])),
                        "cache_dir": str(cache),
                        "cache_key": y["cache_key"],
                        "target": "VAIML",
                    }
                ],
            )
        self.provider = self.session.get_providers()[0]
        self._input = self.session.get_inputs()[0].name

    def infer(self, frame_bgr, conf=None):
        nchw, info = letterbox(frame_bgr, self.imgsz)
        outputs = self.session.run(None, {self._input: nchw})
        return decode_pose(
            outputs,
            info,
            frame_bgr.shape,
            self.conf_threshold if conf is None else conf,
        )


def build_camera(cfg):
    m = cfg["cameras"]["mount"]
    device = m.get("device", 0)
    candidates = [device, 0]
    if isinstance(device, str) and not pathlib.Path(device).exists():
        candidates = [0]  # missing path: skip it quietly
    for candidate in candidates:
        try:
            cam = DirectCamera(
                candidate, m.get("width", 640), m.get("height", 480), m.get("fps", 30)
            )
            print(f"camera: mount camera ({candidate})")
            return cam
        except Exception:
            continue
    print("camera: no camera found - synthetic test pattern")
    return SyntheticCamera()


def build_pose(cfg):
    try:
        from models.vision import PoseEstimator

        pose = PoseEstimator(cfg)  # via models/npu.py build_npu_session
        print(f"pose: YOLOv26s-pose on {pose.provider} (models/npu.py factory)")
        return pose
    except Exception as e:
        reason = e
    try:
        pose = InlinePose(cfg)  # the notebook's own session
        print(
            f"pose: YOLOv26s-pose on {pose.provider} (inline session - "
            f"models/npu.py said: {reason})"
        )
        return pose
    except Exception as e:
        print(f"pose: {e} - scripted person fallback")
        return ScriptedPose()


def build_hands(cfg):
    try:
        with quiet_stderr():  # init chatter lands here, muted
            import mediapipe as mp
            from models.hands import HandTracker

            m = cfg.get("mediapipe", {})
            try:
                hands = HandTracker(cfg)  # via models/hands.py
                which = "models/hands.py tracker"
            except Exception:
                raw = mp.solutions.hands.Hands(  # injected - TODO-independent
                    static_image_mode=False,
                    max_num_hands=int(m.get("max_num_hands", 1)),
                    min_detection_confidence=float(
                        m.get("min_detection_confidence", 0.5)
                    ),
                    min_tracking_confidence=float(
                        m.get("min_tracking_confidence", 0.5)
                    ),
                )
                hands = HandTracker(cfg, hands=raw)
                which = "notebook-injected tracker"
            hands.process(np.zeros((64, 64, 3), dtype=np.uint8))  # silent warmup
        print(f"hands: MediaPipe Hands on CPU ({which})")
        return QuietHands(hands)  # detections log quietly too
    except Exception as e:
        print(f"hands: {e} - scripted fallback")
        return ScriptedHands()


def build_arm(cfg):
    # Same cascade as app.py: the ROS 2 client if a server node is up, else
    # the serial bus directly (this kernel becomes the ONE bus owner), else
    # dry-run. The first two move the REAL arm - keep the space clear.
    if cfg["robot"].get("use_ros2", True) and ros2_available():
        try:
            from ros.arm_client import Ros2ArmClient

            arm = Ros2ArmClient(cfg)
            print("arm: ROS 2 client -> " + cfg["robot"]["command_topic"])
            return arm
        except Exception as e:
            print(f"arm: no ROS 2 server ({e})")
    try:
        arm = make_arm_backend(cfg)
        print("arm: direct serial (sole bus owner!)")
        return arm
    except Exception as e:
        print(f"arm: no hardware ({e})")
        print("arm: dry-run - commands are integrated in software")
        return DryRunArm()


class LockedLiftElbow:
    """Mapper wrapper: shoulder_lift and elbow_flex move as one locked,
    opposed pair (the 02_robot slider link) at half their mapped travel -
    a shoulder deviation of +d from neutral takes the elbow to -d."""

    def __init__(self, mapper, authority=0.5):
        self._m = mapper
        self._authority = float(authority)

    def step(self, *args, **kwargs):
        j = self._m.step(*args, **kwargs)
        sl0 = float(self._m.p.get("shoulder_lift0", -5.0))
        ef0 = float(self._m.p.get("elbow_flex0", 10.0))
        dev = (j["shoulder_lift"] - sl0) * self._authority  # 50% authority
        j["shoulder_lift"] = sl0 + dev
        j["elbow_flex"] = ef0 - dev  # locked, opposite
        return clamp_joints(j)

    def reset(self):
        self._m.reset()

    def __getattr__(self, name):
        return getattr(self._m, name)


ctx = WorkshopContext(
    cfg=cfg,
    arm=build_arm(cfg),
    camera=build_camera(cfg),
    pose=build_pose(cfg),
    hands=build_hands(cfg),
    mapper=LockedLiftElbow(mapper_from_config(cfg)),
    headless=True,
)
print(
    "context ready - arm state:",
    {k: round(v, 1) for k, v in ctx.arm.get_joints().items()},
)

> `app.py` wires this same cascade for the live UI harness. Composition
> code never knows which backend it got: that is what the ABCs bought
> you. Check the printout above - it names exactly what loaded.

## 2. One mimic tick, by hand

This is TODO 3.3, executed step by step. The mapper (provided,
`common.motion.MimicMapper`) carries the real rig's gains. Your job is the
*plumbing between models and mapper*, not the math. With the real camera,
stand where it can see you and keep one hand visible.


In [ ]:
ok, frame = ctx.camera.read()
assert ok, "camera returned no frame - re-run this cell"
h, w = frame.shape[:2]

# 1 eyes: where is the person, which wrist (and its elbow) do we trust?
boxes, scores, kpts = ctx.pose.infer(frame)
wrist = elbow = None
if len(boxes):
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    person = kpts[int(np.argmax(areas))]  # largest box = nearest person
    wrist, elbow = pick_arm(person, ctx.pose.kpt_threshold)  # + same arm's elbow
print(
    "1) wrist (px):",
    None if wrist is None else (round(wrist[0]), round(wrist[1])),
    " elbow:",
    None if elbow is None else (round(elbow[0]), round(elbow[1])),
)

# 2 fingers: depth / roll / pinch
obs, _ = ctx.hands.process(frame)
if obs is None:
    print("2) no hand in view - depth/roll/pinch hold their previous values")
else:
    print(
        f"2) pinch={obs.pinch:.2f}  hand_size={obs.hand_size:.3f}  "
        f"roll_ti={obs.roll_ti:+.1f}"
    )

# 3 touch: is a grip-lock active? (set by the grip behavior in the full pipeline)
gripper_override = (
    float(ctx.shared_state["grip_lock_pos"])
    if ctx.shared_state.get("grip_lock")
    else None
)
print("3) grip-lock override:", gripper_override)

# 4 map -> command -> arm ... unless there is nothing to map (rule 2 below:
# no signals, no command - the arm holds instead of chasing stale smoothing).
if wrist is None and obs is None:
    print("4) no wrist, no hand -> send NOTHING (the arm holds)")
else:
    joints = ctx.mapper.step(
        wrist,
        w,
        h,
        elbow_xy=elbow,  # forearm angle -> wrist_flex
        hand_size=obs.hand_size if obs else None,
        roll_ti=obs.roll_ti if obs else None,
        roll_ref=obs.roll_ref if obs else None,
        roll_ti_weight=obs.roll_ti_weight if obs else 0.0,
        roll_fallback=obs.roll if obs else None,
        pinch=obs.pinch if obs else None,
        gripper_override=gripper_override,
    )
    ctx.arm.send_joints(joints)
    print("4) commanded:", {k: round(v, 1) for k, v in joints.items()})

Four numbered steps. That's the whole TODO. Two rules the harness (and
physics) will hold you to:

- **Infer once per frame.** Pose inference is the expensive step; run it a
  second time for the overlay and you halve your frame rate.
- **No signals, no command.** If neither a wrist nor a hand is visible,
  send *nothing* (the arm holds); commanding from stale smoothing is how
  arms jump.

## 3. Run it as a loop - in simulation

Now the same tick, over and over - in **simulation**, and labeled as one.
A scripted person (wrist orbiting, hand pinching rhythmically) stands in
for the camera and both models, and the commands go to the arm from
section 1: **the real arm follows the simulation**, tracing the scripted
orbit inside the software joint limits. The live view streams the
scripted scene with its skeleton next to the commanded joint traces; it
runs until you press **stop**, then it also plots the full recording. The
joint trajectories tell you instantly whether the composition works - and
the arm shows you the same thing in metal.

The arm moves on its own the whole time this cell runs. Keep the space
around it clear.


In [ ]:
# SIMULATION - a scripted person + scripted hand drive the SAME composition,
# and the commands go to the REAL arm from section 1: the robot follows
# the simulation. Only the scene is synthetic - no camera involved.
# (project/integration/mimic.py's mimic_tick raises until you implement
# TODO 3.3, so the notebook runs the same steps inline.)

sim = WorkshopContext(
    cfg=cfg,
    arm=ctx.arm,  # the REAL arm follows the simulation
    pose=ScriptedPose(),  # a scripted person whose wrist orbits the frame
    hands=ScriptedHands(),  # scripted pinch/roll oscillations
    camera=SyntheticCamera(),  # synthetic scene - no camera involved
    mapper=LockedLiftElbow(mapper_from_config(cfg)),  # same locked pair
    headless=True,
)


def tick_inline(ctx, frame):
    """One mimic tick; returns (joints or None, pose outputs, hand outputs).
    Both models' outputs ride along so the live view can draw the skeleton
    AND the hand landmarks from the SAME inference (rule: infer once per
    frame)."""
    boxes, scores, kpts = ctx.pose.infer(frame)
    wrist = elbow = None
    if len(boxes):
        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        wrist, elbow = pick_arm(kpts[int(np.argmax(areas))], ctx.pose.kpt_threshold)
    obs, raw = ctx.hands.process(frame)
    if wrist is None and obs is None:
        return None, (boxes, scores, kpts), (obs, raw)
    h, w = frame.shape[:2]
    joints = ctx.mapper.step(
        wrist,
        w,
        h,
        elbow_xy=elbow,
        hand_size=obs.hand_size if obs else None,
        roll_ti=obs.roll_ti if obs else None,
        roll_ref=obs.roll_ref if obs else None,
        roll_ti_weight=obs.roll_ti_weight if obs else 0.0,
        roll_fallback=obs.roll if obs else None,
        pinch=obs.pinch if obs else None,
    )
    ctx.arm.send_joints(joints)
    return joints, (boxes, scores, kpts), (obs, raw)


def to_jpeg(img):
    ok, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return jpg.tobytes()


def stamp_simulation(img):
    """Say what this is, on the picture itself: nothing here is real."""
    cv2.rectangle(img, (0, 0), (img.shape[1], 30), (24, 24, 24), -1)
    cv2.putText(
        img,
        "SIMULATION - scripted person driving the real arm",
        (10, 21),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 215, 255),
        2,
    )
    return img


TRACE_JOINTS = ("shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_roll", "gripper")
TRACE_COLORS = (
    (255, 200, 0),
    (80, 220, 80),
    (80, 160, 255),
    (0, 200, 255),
    (200, 120, 255),
)


def draw_joint_traces(history, width=400, height=480, window=240, lo=-110.0, hi=110.0):
    """The commanded joints as a live cv2 strip chart (fast enough to
    redraw every tick, unlike a matplotlib figure)."""
    panel = np.full((height, width, 3), 24, dtype=np.uint8)
    cv2.putText(
        panel,
        "commanded joints - SIMULATION (deg / gripper open %)",
        (10, 20),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.38,
        (170, 170, 170),
        1,
    )
    y0, y1 = 32, height - 52
    zero = int(y1 - (0.0 - lo) / (hi - lo) * (y1 - y0))
    cv2.line(panel, (8, zero), (width - 8, zero), (60, 60, 60), 1)
    data = history[-window:]
    for name, color in zip(TRACE_JOINTS, TRACE_COLORS):
        pts = []
        for k, j in enumerate(data):
            if not j:
                continue
            v = max(lo, min(hi, float(j[name])))
            x = 8 + int(k * (width - 16) / max(window - 1, 1))
            pts.append((x, int(y1 - (v - lo) / (hi - lo) * (y1 - y0))))
        if len(pts) >= 2:
            cv2.polylines(panel, [np.array(pts, np.int32)], False, color, 2)
    x, y = 10, height - 30
    for i, (name, color) in enumerate(zip(TRACE_JOINTS, TRACE_COLORS)):
        cv2.putText(panel, name, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.38, color, 1)
        x += 9 * len(name) + 12
        if i == 2:  # wrap the legend onto a second line
            x, y = 10, height - 12
    return panel


stop_btn = widgets.Button(description="stop", button_style="danger")
stopped = {"flag": False}
stop_btn.on_click(lambda _: stopped.update(flag=True))

# Two persistent Image widgets updated in place (flicker-free): the
# simulated scene the models "see", and what the software arm is told.
cam_view = widgets.Image(format="jpeg", width=430)
trace_view = widgets.Image(format="jpeg", width=340)
display(widgets.VBox([stop_btn, widgets.HBox([cam_view, trace_view])]))

history = []
with ui_events() as poll:
    while not stopped["flag"]:
        poll(10)  # let the stop click arrive
        ok, frame = sim.camera.read()
        if not ok:
            time.sleep(0.05)
            continue
        joints, (boxes, scores, kpts), _hands = tick_inline(sim, frame)
        history.append(joints)
        vis = draw_skeleton(frame, boxes, scores, kpts, sim.pose.kpt_threshold)
        cam_view.value = to_jpeg(stamp_simulation(vis))
        if len(history) % 2 == 0:  # trace at half rate: the browser has
            trace_view.value = to_jpeg(  # to keep up with BOTH streams
                draw_joint_traces(history, height=frame.shape[0])
            )
        time.sleep(0.04)  # ~25 fps: smooth, and no frontend backlog

valid = [j for j in history if j]
if not valid:
    print("no ticks recorded - run the cell again and let it spin for a moment")
else:
    arr = {k: [j[k] for j in valid] for k in valid[0]}
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for k in ("shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_roll", "gripper"):
        ax.plot(arr[k], label=k)
    ax.set_xlabel("tick")
    ax.set_ylabel("degrees / gripper open %")
    ax.legend(ncol=3, fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_title(
        f"mimic composition over {len(valid)} ticks (SIMULATION): "
        "scripted wrist orbit -> joint trajectories"
    )
    plt.show()

Read the plot like an engineer: `shoulder_pan` swings with the scripted
wrist's orbit, `gripper` pulses with the scripted pinch, and everything is
*smooth*. That's the One Euro filter inside the mapper, not luck.

## 4. The UI you'll live in - live

The real stack from section 1, end to end, exactly what `app.py` wraps:
the mount camera through YOLO-pose and MediaPipe hands, the mapper, and
the REAL arm following you. Three live panels: on the left the camera
with BOTH models drawn on the same frame - the YOLO-pose skeleton and the
MediaPipe hand landmarks with the thumb-index pinch line - then the
operator hand/arm the commands mirror in the middle, and the schematic
SO-101 on the right. Stand in view, move your wrist, pinch - the gripper
follows, and you can see exactly what each model saw when it did. Press **stop** when you're done (the arm holds its
last pose; the housekeeping cell at the end parks it).

**This loop commands the real arm.** Keep the space around it clear.


In [ ]:
def draw_hand_arm_panel(joints, width=300, height=480):
    """A picture of the operator's arm and hand, driven by the commanded
    joints: the arm leans with shoulder_lift and bends with elbow_flex, the
    hand tilts with wrist_flex, the finger fan skews with wrist_roll, and
    the thumb-index spread tracks the gripper (pinched shut = closed)."""
    panel = np.full((height, width, 3), 24, dtype=np.uint8)
    j = {
        k: float(joints.get(k, 0.0))
        for k in (
            "shoulder_pan",
            "shoulder_lift",
            "elbow_flex",
            "wrist_flex",
            "wrist_roll",
            "gripper",
        )
    }
    SKIN, EDGE, NAIL = (150, 178, 224), (96, 120, 165), (205, 220, 242)

    cv2.putText(
        panel,
        "the hand it mirrors",
        (12, 22),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (160, 160, 160),
        1,
    )

    def rot(vec, ang):
        c, s = np.cos(ang), np.sin(ang)
        return np.array([vec[0] * c - vec[1] * s, vec[0] * s + vec[1] * c])

    def limb(a, b, wa, wb):
        """A tapered limb segment with rounded joints."""
        a, b = np.asarray(a, float), np.asarray(b, float)
        d = b - a
        n = np.array([-d[1], d[0]]) / max(np.hypot(*d), 1e-6)
        quad = np.array([a + n * wa, b + n * wb, b - n * wb, a - n * wa], np.int32)
        cv2.fillPoly(panel, [quad], SKIN)
        cv2.polylines(panel, [quad], True, EDGE, 2)
        for p, w in ((a, wa), (b, wb)):
            cv2.circle(panel, tuple(p.astype(int)), int(w), SKIN, -1)
            cv2.circle(panel, tuple(p.astype(int)), int(w), EDGE, 2)

    open_frac = float(np.clip(j["gripper"] / 100.0, 0.0, 1.0))

    def finger(base, direction, lengths, w):
        p = np.asarray(base, float)
        d = np.asarray(direction, float)
        d = d / max(np.hypot(*d), 1e-6)
        curl = np.radians(8.0 + 16.0 * (1.0 - open_frac))  # curls as it closes
        for L in lengths:
            q = p + L * d
            cv2.line(
                panel,
                tuple(p.astype(int)),
                tuple(q.astype(int)),
                EDGE,
                w + 4,
                cv2.LINE_AA,
            )
            cv2.line(
                panel, tuple(p.astype(int)), tuple(q.astype(int)), SKIN, w, cv2.LINE_AA
            )
            p, d = q, rot(d, curl)
        cv2.circle(panel, tuple(p.astype(int)), w // 2 + 2, NAIL, -1)

    # --- the arm: shoulder -> elbow -> wrist ------------------------------
    shoulder = np.array([width * 0.18, height * 0.90])
    a1 = np.radians(65.0 + 0.35 * j["shoulder_lift"])  # leans with the lift
    elbow = shoulder + 92.0 * np.array([np.cos(a1), -np.sin(a1)])
    a2 = a1 + np.radians(0.45 * (j["elbow_flex"] - 45.0))  # bends with the elbow
    wrist = elbow + 80.0 * np.array([np.cos(a2), -np.sin(a2)])
    limb(shoulder, elbow, 22, 16)
    limb(elbow, wrist, 16, 12)

    # --- the hand ----------------------------------------------------------
    a3 = a2 + np.radians(0.35 * j["wrist_flex"])  # hand tilt
    u = np.array([np.cos(a3), -np.sin(a3)])  # along the hand
    v = np.array([-u[1], u[0]])  # across the palm
    roll = np.radians(0.35 * j["wrist_roll"])  # roll skews the fan
    palm_c = wrist + 30.0 * u
    cv2.ellipse(
        panel, tuple(palm_c.astype(int)), (28, 22), -np.degrees(a3), 0, 360, SKIN, -1
    )
    cv2.ellipse(
        panel, tuple(palm_c.astype(int)), (28, 22), -np.degrees(a3), 0, 360, EDGE, 2
    )

    # four fingers fan out past the palm (index nearest the thumb side)
    knuckles = palm_c + 18.0 * u
    for off in (-1.5, -0.5, 0.5, 1.5):
        base = knuckles + off * 11.0 * v
        finger(base, rot(u, roll * 0.4 + np.radians(off * 4.0)), (24, 18, 13), 9)

    # the thumb swings toward the index as the gripper closes
    thumb_base = palm_c - 22.0 * v + 4.0 * u
    thumb_dir = rot(u, -np.radians(15.0 + 55.0 * open_frac) + roll * 0.4)
    finger(thumb_base, thumb_dir, (24, 18), 10)

    grip_label = (
        "closed"
        if j["gripper"] <= 45.0
        else ("open" if j["gripper"] >= 90.0 else f"{j['gripper']:.0f}%")
    )
    cv2.putText(
        panel,
        f"pinch -> gripper: {grip_label}",
        (12, height - 14),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (200, 200, 200),
        1,
    )
    return panel


# LIVE: the same loop app.py runs - camera -> models -> mapper -> REAL arm,
# streaming all three panels. tick_inline (section 3) is generic over the
# context, so the exact same composition code now drives the real stack.
stop_btn = widgets.Button(description="stop", button_style="danger")
stopped = {"flag": False}
stop_btn.on_click(lambda _: stopped.update(flag=True))
ui_view = widgets.Image(format="jpeg", width=940)
display(widgets.VBox([stop_btn, ui_view]))

last_joints = ctx.arm.get_joints()
n, t0 = 0, time.time()
with ui_events() as poll:
    while not stopped["flag"]:
        poll(10)  # let the stop click arrive
        ok, frame = ctx.camera.read()
        if not ok:
            time.sleep(0.05)
            continue
        joints, (boxes, scores, kpts), (obs, raw) = tick_inline(
            ctx, frame
        )  # -> REAL arm
        if joints:
            last_joints = joints
        # BOTH models on the SAME frame: the YOLO skeleton, then MediaPipe's
        # hand landmarks + pinch line - each drawn from the one inference
        # this tick already ran.
        vis = draw_skeleton(frame, boxes, scores, kpts, ctx.pose.kpt_threshold)
        if hasattr(ctx.hands, "draw"):
            vis = ctx.hands.draw(vis, obs, raw)
        annotate_lines(
            vis,
            [
                f"gesture mimic - pose: "
                f"{getattr(ctx.pose, 'provider', type(ctx.pose).__name__)}"
                " + MediaPipe hands",
                "wrist x -> shoulder_pan   [TODO 3.3]",
                "pinch   -> gripper        [TODO 3.3]",
            ],
            scale=0.55,
        )
        hand_panel = draw_hand_arm_panel(last_joints, width=300, height=vis.shape[0])
        robot_panel = draw_arm_panel(last_joints, width=300, height=vis.shape[0])
        ui_view.value = to_jpeg(np.hstack([vis, hand_panel, robot_panel]))
        n += 1
dt = max(time.time() - t0, 1e-6)
print(
    f"stopped: {n} frames in {dt:.1f} s = {n/dt:.1f} fps - the arm holds its last pose"
)

## 5. Dispatch: sentence -> behavior (TODOs 3.1, 3.2)

The **registry** keeps the dispatch loop separate from the behaviors, so
deciding *what* a command means stays apart from carrying it *out*.

Each handler registers itself against a specific intent. Because of that, the
loop never turns into a long if/elif chain. To add a new command to the real
pipeline, you need three things: an enum member, a grammar word, and one
registered handler. The loop file itself never changes.

The **gauntlet** in `handle_transcript` is ordered, and the order is a
safety property:

1. `is_stop(raw text)` -> set `ctx.stop_event`, return. *(Before the LLM,
   always. You proved why in the intent-node notebook.)*
2. `is_noise(text)` -> drop. *(Junk transcripts must not move metal.)*
3. `parse(text)` -> `ParsedCommand`.
4. UNKNOWN -> tell the user; no handler -> tell the user; otherwise **clear
   the stop event, run, clear again**. A stale stop from the last behavior
   must never kill the next one.

Watch it work end to end, with a toy registry and the keyword brain:

In [ ]:
def parse(text):
    d = offline_parse_text(text)
    params = {"object": d["object"]} if d.get("object") else {}
    return ParsedCommand(
        Intent.from_string(d["command"]), params=params, transcript=text
    )


registry = CommandRegistry()


@registry.register(Intent.WAVE)
def toy_wave(ctx, cmd):
    print(f"    [toy_wave] waving! (stop_event={ctx.stop_event.is_set()})")
    ctx.arm.send_joints({"wrist_roll": -45.0})
    ctx.arm.send_joints({"wrist_roll": 45.0})


def toy_handle(ctx, text):
    print(f"  heard: {text!r}")
    if is_stop(text):
        ctx.stop_event.set()
        print("    -> STOP fast path (no LLM)")
        return
    if is_noise(text):
        print("    -> dropped by the noise gate")
        return
    cmd = parse(text)
    handler = registry.get(cmd.intent)
    if cmd.intent is Intent.UNKNOWN or handler is None:
        print(f"    -> no behavior for {cmd.intent.value!r}")
        return
    ctx.stop_event.clear()
    handler(ctx, cmd)
    ctx.stop_event.clear()


toy_handle(ctx, "wave at me")
toy_handle(ctx, "STOP right there")
print("    stop_event now set:", ctx.stop_event.is_set())
toy_handle(ctx, "Thank you.")
toy_handle(ctx, "fold my laundry")

Now type your own. This little console feeds whatever you enter through
the same `toy_handle` gauntlet (try `wave at me`, `STOP`, `Thank you.`,
or something it has no behavior for). The cell keeps running so the
buttons stay live; press **done** to release it and move to the next
cell.

In [ ]:
cmd_box = widgets.Text(
    placeholder="type a command, e.g. wave at me",
    continuous_update=False,  # sync the value once, on Enter/blur
    layout=widgets.Layout(width="420px"),
)
send_btn = widgets.Button(description="send", button_style="primary")
done_btn = widgets.Button(description="done", button_style="danger")
console = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="6px"))
finished = {"flag": False}
done_btn.on_click(lambda _: finished.update(flag=True))


def send_command(_=None):
    text = cmd_box.value.strip()
    if not text:
        return
    cmd_box.value = ""
    with console:
        toy_handle(ctx, text)


send_btn.on_click(send_command)
# Enter (or clicking away) commits the text; the observer sends it.
cmd_box.observe(send_command, "value")
display(widgets.VBox([widgets.HBox([cmd_box, send_btn, done_btn]), console]))

with ui_events() as poll:
    while not finished["flag"]:
        poll(10)  # keep send/done clicks flowing
        time.sleep(0.05)
print("console closed")

## 6. Dance: a behavior is a polite loop (TODO 3.4)

The choreography (`common.motion.DANCE_MOVES`) is data. The behavior is
control flow, and its manners are the lesson:

- a **beat clock** (`60/bpm`) paces `arm.go_to(keyframe, duration_s=beat*0.9)`;
- **`ctx.stop_check` is polled between keyframes AND passed into `go_to`**
  so a "stop" interrupts mid-move, not at the end of the song;
- the arm **always parks** (`go_to_rest()` in a `finally`).

The arm is whatever section 1 built. With real hardware the choreography
plays out in metal - **clear the space around the arm** - and `go_to`
works the way the pipeline drives it in direct-goal mode: each keyframe is
sent ONCE and the servo's onboard profile produces the move (re-writing
goals mid-move restarts the profile; that was the 02_robot lesson). On a
dry-run arm the same code just integrates in software and you watch the
printed beats.

The cell below runs a while loop that keeps its buttons live: **dance**
starts the choreography, **stop** interrupts it mid-move (and the arm
still parks: that's the manners), and **done** ends the loop so you can
move to the next cell.


In [ ]:
BEAT_S = 0.6  # fixed beat duration in seconds

beats_slider = widgets.IntSlider(
    value=8, min=2, max=32, description="beats", continuous_update=False
)
dance_btn = widgets.Button(description="dance", button_style="success")
stop_btn = widgets.Button(description="stop", button_style="danger")
done_btn = widgets.Button(description="done")
out = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="6px"))

wants = {"dance": False, "done": False}
dance_btn.on_click(lambda _: wants.update(dance=True))
stop_btn.on_click(lambda _: ctx.stop_event.set())
done_btn.on_click(lambda _: wants.update(done=True))
display(
    widgets.VBox(
        [
            widgets.HBox([beats_slider]),
            widgets.HBox([dance_btn, stop_btn, done_btn]),
            out,
        ]
    )
)

with ui_events() as poll:

    def pump(n=20):
        poll(n)  # push queued widget clicks through now

    def polling_stop_check():
        pump()
        return ctx.stop_event.is_set()

    while not wants["done"]:
        pump()
        if wants["dance"]:
            wants["dance"] = False
            out.clear_output()
            pump()  # flush any clicks queued before this run
            ctx.stop_event.clear()  # then start from a clean flag
            beat = BEAT_S
            sway = DANCE_MOVES["sway"]
            moves = (sway * (int(beats_slider.value) // len(sway) + 1))[
                : int(beats_slider.value)
            ]
            with out:
                print(f"{beat:.2f} s/beat, {len(moves)} beats")
            try:
                for i, keyframe in enumerate(moves):
                    if polling_stop_check():
                        with out:
                            print("  stop! interrupting mid-dance")
                        break
                    ctx.arm.go_to(
                        keyframe,
                        duration_s=beat * 0.9,
                        fps=20,
                        stop_check=polling_stop_check,
                    )
                    if polling_stop_check():  # catch a stop that landed during the move
                        with out:
                            print("  stop! interrupting mid-dance")
                        break
                    j = ctx.arm.get_joints()
                    with out:
                        print(
                            f"  beat {i+1}: pan={j['shoulder_pan']:+6.1f}  "
                            f"roll={j['wrist_roll']:+6.1f}"
                        )
            finally:
                ctx.arm.go_to_rest(duration_s=0.5)  # ALWAYS park
                with out:
                    print(
                        "parked:",
                        {k: round(v, 1) for k, v in ctx.arm.get_joints().items()},
                    )
        time.sleep(0.05)
print("dance controls closed")

In [ ]:
# housekeeping: park the arm, then release every backend (the serial bus
# or ROS client, the camera, MediaPipe) so app.py can take over cleanly.
ctx.stop_event.set()  # end anything still polling stop_check
try:
    ctx.arm.go_to_rest(duration_s=2.0)
finally:
    for closer in (
        getattr(ctx.camera, "close", None),
        getattr(ctx.hands, "close", None),
        getattr(ctx.arm, "close", None),
    ):
        try:
            if closer:
                closer()
        except Exception:
            pass
print("arm parked, backends released")

## 7. Your turn, then the payoff

Open **`project/integration/`** and implement:

| TODO | What | Taught in |
|---|---|---|
| **3.1** | `build_registry()`: register mimic/dance/wave/home | section 5 |
| **3.2** | `handle_transcript()`: the four-step gauntlet | section 5 |
| **3.3** | `mimic_tick()`: the four-step composition | sections 2-3 |
| **3.4** | `run_dance()`: beat clock + stop manners | section 6 |

Then, from the project directory

```bash
python -m integration.selftest
```

Then launch the component-3 harness, the live UI:

```bash
# anywhere (no hardware at all):
python app.py --dry-run --synthetic --no-llm

# the rig (NPU pose, MediaPipe hands, ROS 2 arm, Llama brain):
python app.py  # if all TODOs are implemented, this should run the pipeline fully
```

If the pipeline does not run, go into the solution directory and run,
```bash
python app.py
```
To check your work against the solution. You may use Meld as a diff tool to check your work against the solution.

Webcam + skeleton on the left, the schematic SO-101 mirroring you on the
right, HUD lines naming which TODO feeds which joint, and the real arm
following your hand. Type `dance`, `wave at me`, `go home`, or `stop` into
the terminal while it runs: that's your dispatch gauntlet working.

When the diff between your project and `solution/` is down to style:
congratulations. You've stood up a working vision-language-action pipeline
on an APU.